In [9]:
import pickle
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression


In [10]:
SENTENCES = "../data/sentences.csv.zip"
OUTPUT_FOLDER = "../submission"
ESTIMATOR_PATH = f"{OUTPUT_FOLDER}/estimator.pkl"

In [11]:
def load_data():
    dataframe = pd.read_csv(
        SENTENCES,
        index_col=False,
        compression="zip",
    )

    data = dataframe.phrase
    target = dataframe.target

    return data, target

In [12]:
def make_train_test_split(x, y):
    (x_train, x_test, y_train, y_test) = train_test_split(
        x,
        y,
        test_size=0.25,
        random_state=123456,
    )
    return x_train, x_test, y_train, y_test

In [13]:
def make_pipeline(estimator):
    vectorizer = CountVectorizer(
        lowercase=True,
        analyzer="word",
        token_pattern=r"\b[a-zA-Z]\w+\b",
        stop_words="english",
    )

    transformer = TfidfTransformer()

    pipeline = Pipeline(
        steps=[
            ("vectorizer", vectorizer),
            ("transformer", transformer),
            ("estimator", estimator),
        ],
        verbose=False,
    )

    return pipeline  

In [14]:
def save_estimator(estimator):
    with open(ESTIMATOR_PATH, "wb") as file:
        pickle.dump(estimator, file)

In [15]:
def train_logistic_regression():
    data, target = load_data()
    x_train, _, y_train, _ = make_train_test_split(
        x=data,
        y=target,
    )
    estimator = make_pipeline(estimator=LogisticRegression(max_iter=1000))
    estimator.fit(x_train, y_train)
    save_estimator(estimator)


train_logistic_regression()

In [16]:
def use_estimator():
    dataframe = pd.read_csv(
        SENTENCES,
        index_col=False,
        compression="zip",
    )

    data = dataframe.phrase

    with open(ESTIMATOR_PATH, "rb") as file:
        estimator = pickle.load(file)

    prediction = estimator.predict(data)

    return prediction


use_estimator()

array(['neutral', 'positive', 'positive', ..., 'neutral', 'negative',
       'negative'], dtype=object)